[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/14_kv_cache.ipynb)

# 🔴 Hard: KV Cache Attention

Implement **multi-head attention with KV caching** for efficient autoregressive generation.

During LLM inference, recomputing all key/value projections at every step is wasteful.
A **KV cache** stores previously computed K and V tensors so only the new token(s) need projection.

### Signature
```python
class KVCacheAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x: torch.Tensor, cache=None) -> tuple[torch.Tensor, tuple]:
        # x: (B, S_new, D) — new tokens
        # cache: None or (K_past, V_past) each (B, num_heads, S_past, d_k)
        # Returns: (output, (K_all, V_all))
```

### Requirements
- Inherit from `nn.Module`
- `self.W_q`, `self.W_k`, `self.W_v`, `self.W_o`: `nn.Linear` projections
- When `cache=None` (prefill): apply **causal mask**, return all K/V as cache
- When `cache` provided (decode): concat new K/V with cached, no causal mask needed for single-token decode
- Incremental decode must produce **identical** results to full forward pass

### Key Idea
```
Prefill:  [t0 t1 t2 t3] → full causal attention → cache = (K_{0:3}, V_{0:3})
Decode:   [t4]           → Q=t4, K/V=cache+t4  → cache = (K_{0:4}, V_{0:4})
Decode:   [t5]           → Q=t5, K/V=cache+t5  → cache = (K_{0:5}, V_{0:5})
```

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.9 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import math

In [18]:
from os import W_OK
# ✏️ YOUR IMPLEMENTATION HERE

class KVCacheAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        # pass  # Initialize W_q, W_k, W_v, W_o

        self.W_q=nn.Linear(d_model,d_model)
        self.W_k=nn.Linear(d_model,d_model)
        self.W_v=nn.Linear(d_model,d_model)
        self.W_o=nn.Linear(d_model,d_model)
        self.num_heads=num_heads
        self.d_model=d_model
        self.d_k=self.d_model//self.num_heads


    def forward(self, x, cache=None):

        B,seq,_=x.shape

        q=self.W_q(x)
        k=self.W_k(x)
        v=self.W_v(x)

        # q,k to multiheads


        q=q.view(B,-1,self.num_heads,self.d_k).transpose(1,2)
        k=k.view(B,-1,self.num_heads,self.d_k).transpose(1,2)
        v=v.view(B,-1,self.num_heads,self.d_k).transpose(1,2)

        if cache is not None:
          k_old,v_old=cache
          k= torch.cat([k_old,k], dim=2)
          v=torch.cat([v_old,v],dim=2)

        S_all=k.size(2)
        new_cache=(k,v)

        attention_scores=torch.matmul(q,k.transpose(-1,-2))/math.sqrt(self.d_k)

        if seq>1:
          mask=torch.triu(torch.ones(seq,S_all),diagonal=1).bool()
          attention_scores=attention_scores.masked_fill_(mask,float("-inf"))

        attention_scores=torch.softmax(attention_scores,dim=-1)

        output=torch.matmul(attention_scores,v)
        output=output.transpose(1,2).contiguous().view(B,-1,self.d_model)
        return self.W_o(output),new_cache






        # 1. Project Q, K, V from x
        # 2. Reshape to multi-head: (B, num_heads, S, d_k)
        # 3. If cache exists, concat new K/V with cached K/V
        # 4. Compute attention (causal mask needed during prefill)
        # 5. Return (output, (K_all, V_all))


In [16]:
# 🧪 Debug
torch.manual_seed(0)
attn = KVCacheAttention(d_model=64, num_heads=4)
x = torch.randn(1, 6, 64)

# Full forward
full_out, _ = attn(x)
print("Full output shape:", full_out.shape)  # (1, 6, 64)

# Incremental: prefill 4, decode 1, decode 1
out1, cache = attn(x[:, :4])
print("Cache K shape:", cache[0].shape)  # (1, 4, 4, 16)
out2, cache = attn(x[:, 4:5], cache=cache)
out3, cache = attn(x[:, 5:6], cache=cache)
inc_out = torch.cat([out1, out2, out3], dim=1)
print("Match:", torch.allclose(full_out, inc_out, atol=1e-5))

Full output shape: torch.Size([1, 6, 64])
Cache K shape: torch.Size([1, 4, 4, 16])
Match: False


In [19]:
# ✅ SUBMIT
from torch_judge import check
check('kv_cache')


🧪 Testing: KV Cache Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (no cache) (5.3ms)
  ✅ [2/5] Cache structure (1.9ms)
  ✅ [3/5] Decode step appends to cache (1.9ms)
  ✅ [4/5] Incremental decode matches full forward (2.4ms)
  ✅ [5/5] Gradient flow (2.3ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (13.8ms total)
  Progress saved. Run status() to see your dashboard.

